# cancer-recon-apoptosis — Step 2: Cancer-restricted target discovery (Colab)

**Goal:** a ranked shortlist of receptors enriched on CANCER cells vs matched NORMAL cells — feeds Step 3 (specificity audit) and Step 4 (reward).

**Engine:** CELLxGENE Census (data) → cancer-vs-normal receptor differential (Step 2b, this notebook) → **LIANA+** communication annotation (Step 2c, scripts/05, next).

**Runtime:** CPU is fine; **High-RAM** helps for the fetch. **No GPU.**

**Run log:** every script run is teed to `runs/logs/step2_<UTC>_<gitSHA>.log` and auto-downloaded by the last cell — share that file. The SHA links output to the exact code version.

**Order:** Cell 1 clone → 2 install+log → 3 explore → 4 fetch (~30 min, one-time, cached) → 5 receptor shortlist → 6 finalize+download. If data is already fetched, you can skip 4.

## 1. Clone / pull repo (bootstrap)

In [ ]:
#@title Cell 1 — clone / pull repo (idempotent)
print('[CELL 1] ▶ clone or pull repo')
import os
from pathlib import Path
REPO_URL = 'https://github.com/AnshumanAtrey/cancer-recon-apoptosis.git'
REPO_DIR = Path('/content/cancer-recon-apoptosis')
if REPO_DIR.exists():
    print('  pulling latest')
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
print('  cwd =', os.getcwd())
for s in ('scripts/03_census_fetch.py', 'scripts/04_receptor_targets.py', 'scripts/runlog.py'):
    assert (REPO_DIR / s).exists(), f'missing {s}'
print('[CELL 1] ✓ done')

## 2. Start run log + install deps

In [ ]:
#@title Cell 2 — start run log + install (cellxgene-census / scanpy / liana)
import sys
from scripts.runlog import new_log, run_logged, finalize
RUNLOG = new_log('step2', repo_dir='.')      # one log per session; re-run to start fresh
import importlib.util
need = [m for m in ('cellxgene_census', 'scanpy', 'liana') if importlib.util.find_spec(m) is None]
if need:
    run_logged([sys.executable, '-m', 'pip', 'install', '-q', 'cellxgene-census', 'scanpy', 'liana'],
               RUNLOG, label='pip install cellxgene-census scanpy liana')
else:
    print('  deps already installed')
import cellxgene_census, scanpy
print('  cellxgene_census', cellxgene_census.__version__, '| scanpy', scanpy.__version__)
print('[CELL 2] ✓ done')

## 3. EXPLORE — look at the data first

Per tissue: total cells, `disease` values (exact strings), top `cell_type` values, and whether `malignant`/`neoplastic`/`epithelial` cells are annotated. Already confirmed: lung uses `malignant cell`, colon `neoplastic cell`, breast neither (epithelial subtypes) — scripts/04 adapts automatically.

In [ ]:
#@title Cell 3 — explore Census metadata (cheap)
import sys
from scripts.runlog import new_log, run_logged
RUNLOG = globals().get('RUNLOG') or new_log('step2', repo_dir='.')
rc = run_logged([sys.executable, '-u', 'scripts/03_census_fetch.py', 'explore'], RUNLOG)
print(f'[CELL 3] exit={rc}', '✓' if rc==0 else '✗')

## 4. FETCH — tumour + normal AnnData per target

Capped/subsampled (20k cells/condition) → `data/cellxgene/`. **One-time ~30 min** (normal pools are millions of cells). Idempotent — skips existing, so safe to re-run / resume after a disconnect.

In [ ]:
#@title Cell 4 — fetch tumour + normal cells (~30 min, one-time)
import sys
from scripts.runlog import new_log, run_logged
RUNLOG = globals().get('RUNLOG') or new_log('step2', repo_dir='.')
rc = run_logged([sys.executable, '-u', 'scripts/03_census_fetch.py', 'fetch'], RUNLOG)
print(f'[CELL 4] exit={rc}', '✓' if rc==0 else '✗')
!ls -la data/cellxgene 2>/dev/null

## 5. Receptor shortlist — cancer vs normal differential

`scripts/04_receptor_targets.py`: adaptive cancer-cell selection (malignant/neoplastic where labelled, else epithelial lineage) → Wilcoxon cancer-vs-normal-epithelium on the LIANA receptor gene set → per-tissue ranking → pan-cancer aggregate. Saves `data/cellxgene/targets_shortlist.csv`. Fast (~1-2 min). Prints the top-25 receptors and the Step-1 reference (DR5/DR4) ranks.

In [ ]:
#@title Cell 5 — cancer-enriched receptor shortlist
import sys
from scripts.runlog import new_log, run_logged
RUNLOG = globals().get('RUNLOG') or new_log('step2', repo_dir='.')
rc = run_logged([sys.executable, '-u', 'scripts/04_receptor_targets.py'], RUNLOG)
print(f'[CELL 5] exit={rc}', '✓' if rc==0 else '✗')
import pandas as pd
p = 'data/cellxgene/targets_shortlist.csv'
if rc==0 and __import__('os').path.exists(p):
    print('\nTop 20 cancer-enriched receptors:')
    print(pd.read_csv(p).head(20).to_string(index=False))

## 6. Finalize — download the run log

Browser-downloads `runs/logs/step2_<UTC>_<sha>.log` (the full commit-stamped session). Share that file.

In [ ]:
#@title Cell 6 — finalize + download run log
from scripts.runlog import new_log, finalize
RUNLOG = globals().get('RUNLOG') or new_log('step2', repo_dir='.')
finalize(RUNLOG)
!ls -la runs/logs 2>/dev/null

## 7. Next: Step 2c — LIANA+ communication annotation

`scripts/05` (next): runs LIANA+ on tumour vs normal and flags which shortlisted receptors participate in tumour-enriched ligand-receptor communication (Shriya's cell-cell signalling framing). Share the downloaded run log + `targets_shortlist.csv` so we design it from the real ranking.